# CAS Exam 5: Bornhuetter-Ferguson, Benktander Methods, and Cape Cod Methods

Dataset used:
- `chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Build BF and Benktander estimates using `chainladder.Triangle` objects.
- Connect method outputs back to the formula-sheet interpretation.
- Show Benktander convergence to Chain Ladder ultimate as iterations increase.


## Formula Sheet Reference

### Bornhuetter-Ferguson (BF) Method

$$\text{Ultimate}_{BF} = \text{Latest} + \text{Expected Ultimate} \times \% \text{ Unreported}$$

Credibility-weighted form (equivalent):
$$\text{Ultimate}_{BF} = \underbrace{\frac{1}{\text{CDF}}}_{\text{dev. weight}} \times \text{Ultimate}_{CL} + \underbrace{\left(1 - \frac{1}{\text{CDF}}\right)}_{\% \text{ unreported}} \times \text{Expected Ultimate}$$

- Expected Ultimate = Exposure × ECR (expected claim ratio)
- % Unreported = 1 − 1/CDF = BF weight on the a priori
- % Paid = 1/CDF = BF weight on observed emergence

### Benktander Method (Iterated BF)

Each iteration uses the prior iteration's ultimate as the new expected:

$$\text{Ultimate}^{(k+1)} = \text{Latest} + \left(1 - \frac{1}{\text{CDF}}\right) \times \text{Ultimate}^{(k)}$$

Starting point: Ultimate⁽⁰⁾ = Expected (pure a priori)  
After 1 iteration: Benktander ≈ BF  
As k → ∞: Benktander → Chain Ladder

### Cape Cod Method (Self-Calibrating ECR)

$$\text{ECR}_{CC} = \frac{\sum_w \text{Latest Reported}_w}{\sum_w \text{Used-Up Premium}_w}$$

$$\text{Used-Up Premium}_w = \text{On-Level Premium}_w \times \frac{1}{\text{CDF}_w} = \text{On-Level Premium}_w \times \% \text{ Reported}_w$$

$$\text{Ultimate}_{CC,w} = \text{Latest}_w + \text{ECR}_{CC} \times \text{On-Level Premium}_w \times \left(1 - \frac{1}{\text{CDF}_w}\right)$$

### Method Comparison

| Feature | Chain Ladder | BF | Benktander | Cape Cod |
|---|---|---|---|---|
| A priori source | None (pure experience) | External ECR | Previous BF iteration | Self-derived from data |
| Weight on experience | 100% | 1/CDF | Increases with iterations | Varies by maturity |
| Preferred for immature AYs | No (high leverage) | Yes | Yes | Yes (if premium stable) |
| Preferred for mature AYs | Yes | Yes | Yes | Less critical |
| Requires premium/exposure | No | Yes (exposure) | Yes (exposure) | Yes (premium) |

### Key Assumptions

1. Development pattern (CDF) is correctly selected and consistent across AYs
2. ECR / expected claim ratio is a credible prior (BF/Benktander)
3. Premium is on a consistent rate level for Cape Cod
4. Exposure or premium correctly represents relative size of each AY

In [ ]:
from __future__ import annotations

from pathlib import Path

import chainladder as cl
import pandas as pd

ROOT = Path.cwd().resolve()

from reservingengine.reserving import build_exposure_triangle

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
reported_triangle = triangle['Reported Claims']

{'triangle_shape': triangle.shape, 'valuation_date': str(triangle.valuation_date), 'columns': list(triangle.columns)}, reported_triangle


In [ ]:
# Development pattern (reported basis)
selected_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_triangle)
selected_ldf = selected_dev.ldf_.to_frame()
selected_cdf = selected_dev.cdf_.to_frame()

reported_long = reported_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)
latest_age = reported_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_reported = selected_dev.latest_diagonal.to_frame().iloc[:, 0]

selected_ldf, selected_cdf, latest_age.to_frame(name='LatestAge')


## Development Credibility by Accident Year: % Paid and BF A Priori Weight

The BF credibility weight placed on the a priori (`% Unreported = 1 − 1/CDF`) varies by accident year maturity. The calculation below makes this explicit: the most recent (immature) accident years have the highest % unreported and therefore the highest BF weight on the a priori — which is exactly why BF is preferred for those years.

In [ ]:
_model = cl.Chainladder().fit(reported_triangle)
ppr = pd.DataFrame({
    'latest': reported_triangle.latest_diagonal.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0],
    'ultimate': _model.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0],
    'ibnr': _model.ibnr_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0],
})
ppr['percent_paid'] = ppr['latest'] / ppr['ultimate']
ppr['percent_unreported'] = 1.0 - ppr['percent_paid']
ppr['bf_a_priori_weight'] = ppr['percent_unreported']
ppr.style.format({
    'latest': '{:,.0f}',
    'ultimate': '{:,.0f}',
    'ibnr': '{:,.0f}',
    'percent_paid': '{:.1%}',
    'percent_unreported': '{:.1%}',
    'bf_a_priori_weight': '{:.1%}',
})

**EXAM RED FLAG —** For an accident year with % paid = 20% (% unreported = 80%), the BF weight on the a priori is 80%. A 10% error in the ECR produces approximately an 8% error in the BF ultimate for that year. For a mature AY with % paid = 90% (% unreported = 10%), the same 10% ECR error produces only ~1% ultimate error.

This is why ECR quality matters most for the most recent (immature) accident years — exactly the years where you have the least historical data to calibrate the ECR.

**EXAM RED FLAG —** The `bf_a_priori_weight` column is identical to `% unreported`. On the exam, if asked "what weight does BF place on the a priori for AY 2007?", look up the % unreported for that AY.

## Set the Expected Claims Prior

This dataset does not include premium, so we create an explicit prior input for BF/Benktander demonstration:
- Assume a prior ELR by AY to infer an earned-premium proxy from latest reported.
- Select one portfolio-level apriori ELR used by BF and Benktander estimators.

This is the central actuarial judgment in BF/Benktander.


In [ ]:
ay_year = latest_reported.index.year
prior_elr_by_ay = pd.Series(
    [0.720, 0.725, 0.730, 0.735, 0.740, 0.745, 0.750, 0.755, 0.760, 0.765],
    index=ay_year,
    dtype=float,
)

earned_premium_proxy = latest_reported / prior_elr_by_ay.values
selected_apriori = 0.750
expected_ultimate_prior = earned_premium_proxy * selected_apriori

exposure_series = pd.Series(earned_premium_proxy.values, index=ay_year, dtype=float)
sample_weight = build_exposure_triangle(exposure_series, selected_dev)

prior_table = pd.DataFrame(
    {
        'LatestReported': latest_reported.values,
        'PriorELR_ByAY': prior_elr_by_ay.values,
        'EarnedPremiumProxy': earned_premium_proxy.values,
        'SelectedAprioriELR': selected_apriori,
        'ExpectedUltimatePrior': expected_ultimate_prior.values,
    },
    index=ay_year,
)
prior_table.index.name = 'AccidentYear'
prior_table


## BF Estimate: Formula Check and Estimator Output

Reported-basis BF formula used here:
- Ultimate = Latest Reported + Expected Ultimate Prior x (1 - 1/CDF)

Below compares a manual formula implementation to `chainladder.BornhuetterFerguson`.


In [ ]:
bf_model = cl.BornhuetterFerguson(apriori=selected_apriori).fit(selected_dev, sample_weight=sample_weight)
bf_ultimate_model = bf_model.ultimate_.to_frame().iloc[:, 0]

cdf_map = selected_dev.cdf_.to_frame().iloc[0]
selected_cdf_by_ay = latest_age.map(lambda age: cdf_map.get(f'{int(age)}-Ult', 1.0))
bf_ultimate_manual = latest_reported + expected_ultimate_prior.values * (1.0 - 1.0 / selected_cdf_by_ay.values)
bf_ultimate_manual = pd.Series(bf_ultimate_manual, index=latest_reported.index)

bf_check = pd.DataFrame(
    {
        'LatestReported': latest_reported,
        'SelectedCDF': selected_cdf_by_ay.values,
        'ExpectedUltimatePrior': expected_ultimate_prior.values,
        'BF_Ultimate_Manual': bf_ultimate_manual.values,
        'BF_Ultimate_Model': bf_ultimate_model.values,
    },
    index=ay_year,
)
bf_check['AbsDiff_Manual_vs_Model'] = (bf_check['BF_Ultimate_Manual'] - bf_check['BF_Ultimate_Model']).abs()
bf_check


## Benktander Convergence Toward Chain Ladder

Benktander with `n_iters=1` is BF-like.
As `n_iters` increases, the estimate puts more weight on development and converges to Chain Ladder.


In [ ]:
cl_model = cl.Chainladder().fit(selected_dev)
cl_total_ultimate = float(cl_model.ultimate_.sum())

benktander_rows = []
for n_iters in [1, 2, 3, 5, 10, 20, 50]:
    ben_model = cl.Benktander(apriori=selected_apriori, n_iters=n_iters).fit(selected_dev, sample_weight=sample_weight)
    ben_total_ultimate = float(ben_model.ultimate_.sum())
    ben_total_ibnr = float(ben_model.ibnr_.sum())
    abs_diff = abs(cl_total_ultimate - ben_total_ultimate)
    benktander_rows.append(
        {
            'n_iters': n_iters,
            'Benktander_TotalUltimate': ben_total_ultimate,
            'Benktander_TotalIBNR': ben_total_ibnr,
            'ChainLadder_TotalUltimate': cl_total_ultimate,
            'AbsDiff_to_CL': abs_diff,
            'PctDiff_to_CL': abs_diff / cl_total_ultimate,
        }
    )

benktander_convergence = pd.DataFrame(benktander_rows).set_index('n_iters')
benktander_convergence


## AY-Level View: BF, Benktander, and Chain Ladder

This table compares AY ultimates for:
- BF (`n_iters=1` equivalent conceptually),
- Benktander at multiple iterations,
- Chain Ladder target.


In [ ]:
ben_n1 = cl.Benktander(apriori=selected_apriori, n_iters=1).fit(selected_dev, sample_weight=sample_weight)
ben_n3 = cl.Benktander(apriori=selected_apriori, n_iters=3).fit(selected_dev, sample_weight=sample_weight)
ben_n10 = cl.Benktander(apriori=selected_apriori, n_iters=10).fit(selected_dev, sample_weight=sample_weight)

ay_compare = pd.DataFrame(
    {
        'BF_Ultimate': bf_model.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n1': ben_n1.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n3': ben_n3.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n10': ben_n10.ultimate_.to_frame().iloc[:, 0].values,
        'ChainLadder_Ultimate': cl_model.ultimate_.to_frame().iloc[:, 0].values,
    },
    index=ay_year,
)
ay_compare['n1_to_CL_gap'] = ay_compare['ChainLadder_Ultimate'] - ay_compare['Benktander_n1']
ay_compare['n10_to_CL_gap'] = ay_compare['ChainLadder_Ultimate'] - ay_compare['Benktander_n10']
ay_compare.loc['Total'] = ay_compare.sum()
ay_compare


## Cape Cod Method: ECR Derivation and Setup

Cape Cod is closely related to BF, but it derives the expected claim ratio from reported data and used-up premium.

Formula view:
- Cape Cod ECR = Total Reported to Date / Total Used-up Premium
- Total Used-up Premium = sum(On-level Earned Premium x % Reported)

Educational notes:
- Only reported claims are used in the calibration year set.
- Premiums should be adjusted to a consistent on-level basis.


In [ ]:
# On-level premium assumption by AY (example educational setup)
onlevel_factor_by_ay = pd.Series(
    [1.080, 1.070, 1.060, 1.050, 1.040, 1.030, 1.020, 1.010, 1.005, 1.000],
    index=ay_year,
    dtype=float,
)

onlevel_earned_premium = earned_premium_proxy * onlevel_factor_by_ay.values
percent_reported_by_ay = 1.0 / selected_cdf_by_ay.values
used_up_premium = onlevel_earned_premium * percent_reported_by_ay
cape_cod_ecr_manual = float(latest_reported.sum() / used_up_premium.sum())

cape_cod_weight = build_exposure_triangle(
    pd.Series(onlevel_earned_premium.values, index=ay_year, dtype=float),
    selected_dev,
)
cape_cod_model = cl.CapeCod(trend=0.0, decay=1.0, n_iters=1).fit(
    selected_dev,
    sample_weight=cape_cod_weight,
)
cape_cod_ecr_model = float(cape_cod_model.apriori_.to_frame().iloc[0, 0])

cape_cod_setup = pd.DataFrame(
    {
        'LatestReported': latest_reported.values,
        'SelectedCDF': selected_cdf_by_ay.values,
        'PctReported': percent_reported_by_ay,
        'OnLevelFactor': onlevel_factor_by_ay.values,
        'OnLevelEarnedPremium': onlevel_earned_premium.values,
        'UsedUpPremium': used_up_premium,
    },
    index=ay_year,
)
cape_cod_setup.index.name = 'AccidentYear'

pd.Series({
    'CapeCodECR_Manual': cape_cod_ecr_manual,
    'CapeCodECR_ModelApriori': cape_cod_ecr_model,
    'AbsDiff_Manual_vs_Model': abs(cape_cod_ecr_manual - cape_cod_ecr_model),
}), cape_cod_setup

### On-Level Premium: Why It Matters for Cape Cod

The Cape Cod ECR is computed using used-up premium where each AY's premium is weighted by its % reported. If premiums are not on a consistent rate level, the ECR will be distorted.

**Example:** If rates increased 20% in AY 2006, the 2006 premium is 20% larger in dollar terms, but this does not represent 20% more expected claims — the underlying exposure base is the same. The on-level factor restates premium to a consistent rate level before computing Cape Cod.

**Exam formula for on-level adjustment:**

$$\text{On-Level Factor}_w = \frac{\text{Current Rate Level Index}}{\text{Rate Level Index at inception of AY } w}$$

$$\text{Adjusted Premium}_w = \text{As-Written Premium}_w \times \text{On-Level Factor}_w$$

Then substitute Adjusted Premium into the Cape Cod ECR formula.

**When to apply on-level adjustment:**
- Rate levels changed materially (> 5–10%) over the triangle period
- The exam question provides rate change information
- Simple Cape Cod ECR without on-level adjustment would be biased

**EXAM RED FLAG —** If the exam gives you rate changes by accident year, always apply the on-level adjustment to premium before computing the Cape Cod ECR. Failing to do so will produce a biased ECR that overstates the ECR when rates were high and understates when rates were low.

## Cape Cod Results, Comparison, and Impacts

Advantages/disadvantages (formula-sheet style):
- Advantage: compared with pure development, Cape Cod can be more resilient to random AY volatility.
- Disadvantage: requires sufficient credible reported claims and careful on-level premium treatment.

The comparison below shows where Cape Cod lands relative to BF, Benktander, and Chain Ladder in this dataset.


In [ ]:
cape_cod_ultimate = cape_cod_model.ultimate_.to_frame().iloc[:, 0]
cape_cod_ibnr = cape_cod_model.ibnr_.to_frame().iloc[:, 0]

cape_cod_compare = pd.DataFrame(
    {
        'CapeCod_Ultimate': cape_cod_ultimate.values,
        'BF_Ultimate': bf_model.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n3_Ultimate': ben_n3.ultimate_.to_frame().iloc[:, 0].values,
        'ChainLadder_Ultimate': cl_model.ultimate_.to_frame().iloc[:, 0].values,
    },
    index=ay_year,
)
cape_cod_compare['CapeCod_to_CL_gap'] = cape_cod_compare['ChainLadder_Ultimate'] - cape_cod_compare['CapeCod_Ultimate']
cape_cod_compare.loc['Total'] = cape_cod_compare.sum()

cape_cod_totals = pd.Series({
    'CapeCod_TotalUltimate': float(cape_cod_model.ultimate_.sum()),
    'CapeCod_TotalIBNR': float(cape_cod_model.ibnr_.sum()),
    'BF_TotalUltimate': float(bf_model.ultimate_.sum()),
    'Benktander_n3_TotalUltimate': float(ben_n3.ultimate_.sum()),
    'ChainLadder_TotalUltimate': float(cl_model.ultimate_.sum()),
})

cape_cod_impact_table = pd.DataFrame(
    [
        ['Increase in exposure', 'No material effect (as long as average accident date is held constant)'],
        ['Average accident date shifts forward', 'Underestimates ultimate claims by less than development method but more than BF method'],
        ['Increase claim ratios', 'Underestimates ultimate claims unless most recent data is reflected in Cape Cod ECR'],
        ['Speedup in claim settlement rate', 'No material effect'],
        ['Increase in case outstanding adequacy', 'Overestimates ultimate claims by less than development method but more than BF method'],
        ['Change in product mix', 'Accuracy is impacted when lines have different development patterns or ECRs'],
    ],
    columns=['Description', 'Impact on Cape Cod method'],
)

cape_cod_totals, cape_cod_compare, cape_cod_impact_table


## When to Use BF / Benktander / Cape Cod

**Use BF when:**
- Recent accident years are immature (high CDF, low % paid)
- Chain ladder results appear volatile due to sparse early development
- You have a credible a priori ECR from pricing or external benchmarks
- The exam question specifically asks for a method that blends a priori and experience

**Use Benktander when:**
- All BF conditions apply, AND you want to reduce dependence on the a priori as iterations increase
- n_iters = 2 is the most commonly tested Benktander variant on Exam 5
- You want a smoother transition between immature and mature AY treatment

**Use Cape Cod when:**
- You need a self-calibrating ECR from the data itself (no reliable external a priori)
- Premium is available and reasonably on-level (or can be adjusted)
- Sufficient credible reported claims data is available

**Do NOT use BF/Benktander when:**
- The a priori ECR is poorly calibrated — errors propagate directly into reserves for immature AYs
- All accident years are fully mature — chain ladder dominates and BF adds no value

**Do NOT use Cape Cod when:**
- Premium data is unavailable or unreliable
- Rate changes have been extreme and on-level adjustment cannot be applied confidently

**EXAM RED FLAG —** BF is most sensitive to ECR quality for the most immature accident years (highest % unreported = highest a priori weight). A poor ECR for the most recent AY can produce a large reserve error. Always validate the ECR before applying BF to highly immature years.

In [ ]:
impact_table = pd.DataFrame(
    [
        [
            'Increase in exposure',
            'No material effect if ECR held constant',
            'No material effect if ECR held constant',
            'ECR is self-calibrating — may shift if mix changes alongside exposure',
        ],
        [
            'Average accident date shifts forward',
            'Underestimates ultimate by less than development method',
            'Underestimates by slightly less than BF (more weight on experience)',
            'Same as BF',
        ],
        [
            'Increase in claim ratios',
            'Underestimates if ECR not updated; less affected than pure development',
            'Same as BF; attenuated with more iterations',
            'Self-calibrates — ECR will adjust if the increase is within the triangle window',
        ],
        [
            'Speedup in claim settlement rate',
            'Overstates on paid basis by less than development method; minimal on reported basis',
            'Minimal effect on reported basis',
            'Same as BF on respective basis',
        ],
        [
            'Increase in case outstanding adequacy',
            'Minimal effect on paid basis',
            'Minimal effect on paid basis',
            'Overstates on reported basis by less than development method',
        ],
        [
            'Change in product mix',
            'ECR should be re-segmented to reflect new mix',
            'ECR should be re-segmented',
            'Used-up premium calculation needs to reflect new mix exposure',
        ],
        [
            'Poor ECR calibration (a priori wrong)',
            'Error ≈ % Unreported × ECR error — worst for immature AYs',
            'Error attenuated with more iterations (more weight shifts to experience)',
            'Cape Cod self-calibrates — less sensitive to external ECR quality; sensitive to premium quality instead',
        ],
    ],
    columns=['Change in Environment', 'BF Impact', 'Benktander Impact', 'Cape Cod Impact'],
)

summary_totals = pd.Series({
    'CL_TotalUltimate': cl_total_ultimate,
    'BF_TotalUltimate': float(bf_model.ultimate_.sum()),
    'BF_TotalIBNR': float(bf_model.ibnr_.sum()),
    'Benktander_n1_TotalUltimate': float(ben_n1.ultimate_.sum()),
    'Benktander_n10_TotalUltimate': float(ben_n10.ultimate_.sum()),
})

impact_table, summary_totals

---
# Exam 5 Practice Problems — BF, Benktander, and Cape Cod Methods

The following problems mirror Exam 5 written question format.
Work through each calculation before checking the solution code.
All arithmetic is reproducible without code.

**Instructions:** Show all work. Round CDFs and % unreported to 4 decimal places. Round dollar amounts to nearest whole number.


## Practice Problem 1: Pure BF Calculation

**Exam-style question (mechanical, 4–5 points):**

An actuary is estimating reserves for a commercial auto book. The selected ELR is **0.850**. The following data is available:

| AY   | Earned Premium (000s) | Reported to Date (000s) | CDF to Ultimate |
|------|----------------------:|------------------------:|----------------:|
| 2021 | 25,000                | 20,000                  | 1.050           |
| 2022 | 27,000                | 18,000                  | 1.100           |
| 2023 | 29,000                | 15,000                  | 1.200           |
| 2024 | 31,000                | 11,000                  | 1.500           |
| 2025 | 33,000                | 5,600                   | 3.000           |

**(a)** Calculate the Expected Ultimate for each AY. *(1 pt)*  
**(b)** Calculate % Unreported for each AY. *(1 pt)*  
**(c)** Calculate the BF IBNR for each AY using `IBNR = Expected Ultimate × % Unreported`. *(1 pt)*  
**(d)** Calculate the BF Ultimate for each AY. *(1 pt)*  
**(e)** Which AY has the highest BF IBNR as a % of its BF Ultimate? In one sentence, explain why. *(1 pt)*  

> **Key formula:** BF Ultimate = Reported + Expected Ultimate × % Unreported  
> **Common trap:** Do not multiply ELR by Reported — always multiply ELR by **Earned Premium**.


In [ ]:
import pandas as pd
import numpy as np

# PRACTICE PROBLEM 1 SOLUTION: Pure BF Calculation
elr = 0.850

data = {
    'AY':       [2021,  2022,  2023,  2024,  2025],
    'Premium':  [25000, 27000, 29000, 31000, 33000],
    'Reported': [20000, 18000, 15000, 11000,  5600],
    'CDF':      [1.050, 1.100, 1.200, 1.500, 3.000],
}
df = pd.DataFrame(data).set_index('AY')

# PART (a): Expected Ultimate = ELR x Earned Premium
df['Exp_Ultimate'] = df['Premium'] * elr

# PART (b): % Unreported = 1 - 1/CDF  (this is the BF a priori weight)
df['Pct_Unreported'] = 1 - 1 / df['CDF']

# PART (c): BF IBNR = Expected Ultimate x % Unreported
df['BF_IBNR'] = df['Exp_Ultimate'] * df['Pct_Unreported']

# PART (d): BF Ultimate = Reported + BF IBNR
df['BF_Ultimate'] = df['Reported'] + df['BF_IBNR']

# PART (e): BF IBNR as % of BF Ultimate
df['BF_IBNR_Pct'] = df['BF_IBNR'] / df['BF_Ultimate']

result = df.copy()
result.loc['Total'] = {
    'Premium':       df['Premium'].sum(),
    'Reported':      df['Reported'].sum(),
    'CDF':           float('nan'),
    'Exp_Ultimate':  df['Exp_Ultimate'].sum(),
    'Pct_Unreported':float('nan'),
    'BF_IBNR':       df['BF_IBNR'].sum(),
    'BF_Ultimate':   df['BF_Ultimate'].sum(),
    'BF_IBNR_Pct':   df['BF_IBNR'].sum() / df['BF_Ultimate'].sum(),
}

print('=== PRACTICE PROBLEM 1: BF PROJECTION ===')
print(f'  Selected ELR: {elr:.3f}')
print()
print(result.to_string(
    float_format=lambda x: f'{x:.4f}' if 0 < abs(x) < 10 else f'{x:,.0f}'
))
print()

max_ay = df['BF_IBNR_Pct'].idxmax()
print(f'PART (e): AY {max_ay} has the highest BF IBNR as % of BF Ultimate'
      f' = {df.loc[max_ay, "BF_IBNR_Pct"]*100:.1f}%')
print()
print(
    'MODEL ANSWER:\n'
    'AY 2025 has the highest BF IBNR as a % of ultimate because it is at '
    'the earliest development age (% unreported = 66.7%) -- BF relies almost '
    'entirely on the a priori ELR for this year, with very little credibility '
    'given to the sparse actual reported claims.\n'
    '\n'
    'KEY CHECKS:\n'
    '  (1) % Unreported = 1 - 1/CDF  (NOT 1 - CDF)\n'
    '  (2) BF IBNR uses Expected Ultimate, not CL Ultimate\n'
    '  (3) BF Ultimate = Reported + BF IBNR (not just Expected Ultimate)\n'
    '  (4) ELR x Premium, never ELR x Reported'
)


## Practice Problem 2: Benktander Calculation

**Exam-style question (mechanical + conceptual, 5–6 points):**

Using the same data from Practice Problem 1 (ELR = 0.850), and the BF ultimates you computed:

| AY   | Reported (000s) | CDF   | BF Ultimate (000s) |
|------|----------------:|------:|-------------------:|
| 2021 | 20,000          | 1.050 | 21,012             |
| 2022 | 18,000          | 1.100 | 19,827             |
| 2023 | 15,000          | 1.200 | 18,188             |
| 2024 | 11,000          | 1.500 | 17,633             |
| 2025 | 5,600           | 3.000 | 21,400             |

**(a)** Calculate the Chain Ladder ultimate for each AY. *(1 pt)*  
**(b)** Calculate the Benktander ultimate for each AY using one iteration:  
> `Benktander = Reported + % Unreported × BF Ultimate` *(2 pts)*  

**(c)** Order the three methods (CL, BF, Benktander) from **lowest to highest** for AY 2025. Explain why Benktander falls between BF and CL. *(2 pts)*  

**(d)** For AY 2021 (the most mature), all three methods produce nearly identical results. Briefly explain why. *(1 pt)*  

> **Key formula:** Benktander$^{(2)}$ = Reported + (1 − 1/CDF) × BF Ultimate$^{(1)}$  
> **Common trap:** Benktander is NOT a simple average of CL and BF. The weight is % unreported (= 1 − 1/CDF), not 50/50.


In [ ]:
import pandas as pd
import numpy as np

# PRACTICE PROBLEM 2 SOLUTION: Benktander Calculation
elr = 0.850

data = {
    'AY':        [2021,  2022,  2023,  2024,  2025],
    'Premium':   [25000, 27000, 29000, 31000, 33000],
    'Reported':  [20000, 18000, 15000, 11000,  5600],
    'CDF':       [1.050, 1.100, 1.200, 1.500, 3.000],
}
df = pd.DataFrame(data).set_index('AY')

df['Pct_Unreported'] = 1 - 1 / df['CDF']
df['Exp_Ultimate']   = df['Premium'] * elr

# PART (a): Chain Ladder ultimate = Reported x CDF
df['CL_Ultimate'] = df['Reported'] * df['CDF']

# BF Ultimate (from Problem 1)
df['BF_Ultimate'] = df['Reported'] + df['Exp_Ultimate'] * df['Pct_Unreported']

# PART (b): Benktander (1 iteration) = Reported + % Unreported x BF Ultimate
df['BK_Ultimate'] = df['Reported'] + df['Pct_Unreported'] * df['BF_Ultimate']

# Summary
result = df[['Reported', 'CDF', 'Pct_Unreported',
             'CL_Ultimate', 'BF_Ultimate', 'BK_Ultimate']].copy()
result.loc['Total'] = {
    'Reported':      df['Reported'].sum(),
    'CDF':           float('nan'),
    'Pct_Unreported':float('nan'),
    'CL_Ultimate':   df['CL_Ultimate'].sum(),
    'BF_Ultimate':   df['BF_Ultimate'].sum(),
    'BK_Ultimate':   df['BK_Ultimate'].sum(),
}

print('=== PRACTICE PROBLEM 2: CL vs BF vs BENKTANDER ===')
print(result.to_string(
    float_format=lambda x: f'{x:.4f}' if 0 < abs(x) < 10 else f'{x:,.0f}'
))
print()

# Part (c) - AY 2025 ordering
ay = 2025
cl  = df.loc[ay, 'CL_Ultimate']
bk  = df.loc[ay, 'BK_Ultimate']
bf  = df.loc[ay, 'BF_Ultimate']
pct = df.loc[ay, 'Pct_Unreported']
print(f'PART (c): AY {ay} -- % Unreported = {pct:.4f} ({pct*100:.1f}%)')
print(f'  CL  = {cl:>10,.0f}  (Reported x CDF -- highly leveraged)')
print(f'  BK  = {bk:>10,.0f}  (Reported + % Unreported x BF_Ult)')
print(f'  BF  = {bf:>10,.0f}  (Reported + % Unreported x Expected_Ult)')
print(f'  Order: BF < BK < CL  (BK is between because it uses BF_Ult as'
      f' prior, which is higher than Expected_Ult but lower than CL_Ult)')
print()

# Part (d) - AY 2021 convergence
ay2 = 2021
cl2  = df.loc[ay2, 'CL_Ultimate']
bk2  = df.loc[ay2, 'BK_Ultimate']
bf2  = df.loc[ay2, 'BF_Ultimate']
pct2 = df.loc[ay2, 'Pct_Unreported']
print(f'PART (d): AY {ay2} -- % Unreported = {pct2:.4f} ({pct2*100:.1f}%)')
print(f'  CL = {cl2:>8,.0f} | BK = {bk2:>8,.0f} | BF = {bf2:>8,.0f}')
print()
print(
    'MODEL ANSWER (d):\n'
    'At 95.2% paid (CDF = 1.050), AY 2021 has % unreported = 4.8%. '
    'Both BF and Benktander give 95.2% weight to actual reported claims '
    'and only 4.8% weight to the a priori prior. At this maturity, the '
    'a priori assumption has almost no impact -- all three methods converge '
    'to approximately the same answer.\n'
    '\n'
    'MODEL ANSWER (c):\n'
    'BF < Benktander < CL for AY 2025. BF uses the expected ultimate as its '
    'prior; Benktander uses the BF ultimate (which is higher than expected '
    'because actual reported claims are included). CL fully trusts the 3.0x '
    'CDF applied to sparse 12-month data -- it is the highest and most volatile.\n'
    '\n'
    'KEY TRAP: Benktander is NOT a simple average of CL and BF. '
    'The weight applied is always % unreported (= 1 - 1/CDF).'
)


## Practice Problem 3: Method Comparison and Recommendation

**Exam-style question (judgment, 6–7 points):**

An actuary has computed the following ultimates for a workers compensation book. Selected ELR = 0.820. AY 2025 is at 12 months of development.

| AY   | Earned Prem (000s) | Reported (000s) | CDF   | CL Ult (000s) | BF Ult (000s) | BK Ult (000s) |
|------|-------------------:|----------------:|------:|--------------:|--------------:|--------------:|
| 2021 | 40,000             | 35,000          | 1.050 | 36,750        | 36,631        | 36,737        |
| 2022 | 43,000             | 33,000          | 1.110 | 36,630        | 36,171        | 36,583        |
| 2023 | 46,000             | 27,000          | 1.250 | 33,750        | 33,360        | 33,672        |
| 2024 | 50,000             | 19,000          | 1.600 | 30,400        | 30,200        | 30,333        |
| 2025 | 54,000             | 8,100           | 4.000 | 32,400        | 30,528        | 31,092        |

**(a)** Verify the BF Ultimate for AY 2025. Show your work. *(2 pts)*  
**(b)** Verify the Benktander Ultimate for AY 2025. Show your work. *(2 pts)*  
**(c)** For AY 2025, explain in 3–4 sentences why you would recommend Benktander over both pure Chain Ladder and pure BF. *(2 pts)*  
**(d)** For AY 2021, which method would you recommend and why? One sentence. *(1 pt)*  


In [ ]:
import pandas as pd

# PRACTICE PROBLEM 3 SOLUTION: Method Comparison
elr = 0.820

data = {
    'AY':       [2021,  2022,  2023,  2024,  2025],
    'Premium':  [40000, 43000, 46000, 50000, 54000],
    'Reported': [35000, 33000, 27000, 19000,  8100],
    'CDF':      [1.050, 1.110, 1.250, 1.600, 4.000],
}
df = pd.DataFrame(data).set_index('AY')

df['Pct_Unreported'] = 1 - 1 / df['CDF']
df['Exp_Ultimate']   = df['Premium'] * elr
df['CL_Ultimate']    = df['Reported'] * df['CDF']
df['BF_Ultimate']    = df['Reported'] + df['Exp_Ultimate'] * df['Pct_Unreported']
df['BK_Ultimate']    = df['Reported'] + df['Pct_Unreported'] * df['BF_Ultimate']

print('=== FULL METHOD COMPARISON TABLE ===')
cols = ['Reported', 'CDF', 'Pct_Unreported', 'Exp_Ultimate',
        'CL_Ultimate', 'BF_Ultimate', 'BK_Ultimate']
print(df[cols].to_string(
    float_format=lambda x: f'{x:.4f}' if 0 < abs(x) < 10 else f'{x:,.0f}'
))
print()

# Parts (a) & (b): Verify AY 2025
ay = 2025
rep  = df.loc[ay, 'Reported']
cdf  = df.loc[ay, 'CDF']
pct  = df.loc[ay, 'Pct_Unreported']
exp  = df.loc[ay, 'Exp_Ultimate']
bf   = df.loc[ay, 'BF_Ultimate']
bk   = df.loc[ay, 'BK_Ultimate']
cl   = df.loc[ay, 'CL_Ultimate']

print(f'=== PARTS (a) & (b): AY {ay} VERIFICATION ===')
print(f'  Reported:        {rep:>10,.0f}')
print(f'  CDF:             {cdf:>10.3f}')
print(f'  % Unreported:    {pct:>10.4f}  [= 1 - 1/{cdf:.3f}]')
print(f'  Expected Ult:    {exp:>10,.0f}  [= {elr} x {df.loc[ay,"Premium"]:,}]')
print()
print(f'  (a) BF Ultimate: {rep:,.0f} + {pct:.4f} x {exp:,.0f}'
      f' = {rep:,.0f} + {pct*exp:,.0f} = {bf:,.0f}')
print(f'  (b) BK Ultimate: {rep:,.0f} + {pct:.4f} x {bf:,.0f}'
      f' = {rep:,.0f} + {pct*bf:,.0f} = {bk:,.0f}')
print()
print(f'  Method ordering for AY {ay}:')
print(f'    BF  = {bf:>9,.0f}  (lowest -- 100% a priori for unreported)')
print(f'    BK  = {bk:>9,.0f}  (middle  -- prior is BF ult, not just ELR)')
print(f'    CL  = {cl:>9,.0f}  (highest -- Reported x {cdf:.1f}x CDF)')
print()

print('PART (c) -- MODEL ANSWER (exam format):')
print('-' * 60)
print(
    'AY 2025 is at only 12 months of development with % paid = 25%.\n'
    'Chain Ladder applies a 4.0x CDF to very sparse reported claims --\n'
    'this extreme leverage means a small LDF error produces a large\n'
    'ultimate error; pure CL is unreliable here.\n'
    '\n'
    'Pure BF ignores all actual emergence entirely (100% weight on ELR),\n'
    'which also discards the partial signal in $8,100 of reported claims.\n'
    '\n'
    'Benktander is the best choice: it gives 75% weight to the BF\n'
    'ultimate (which already blends a priori and actual data) and 25%\n'
    'to reported claims, producing a more balanced estimate than either\n'
    'extreme without fully trusting the volatile early development.\n'
    '\n'
    'The key caveat: Benktander still relies on the a priori ELR quality;\n'
    'if ELR is significantly wrong, both BF and Benktander will be biased.'
)
print()
print('PART (d) -- MODEL ANSWER:')
print('-' * 60)
print(
    'For AY 2021 (95.2% paid), Chain Ladder is preferred -- at this\n'
    'maturity, actual development is credible and the a priori ELR has\n'
    'negligible impact on BF/Benktander anyway (all three nearly equal).'
)


---
## Sensitivity and Impact Questions — Directional Reference

These questions ask: *"If X changes, what happens to the reserve?"* You must know the **direction** and **which methods are affected**.

### If ELR Increases:

| Method | Impact | Mechanism |
|---|:---:|---|
| **Chain Ladder** | None | CL does not use ELR at all |
| **BF** | Increases materially | IBNR = Expected Ult × % Unreported; higher ELR → higher expected → higher IBNR |
| **Benktander** | Increases (attenuated) | Uses BF ultimate as prior; effect is dampened by one iteration |
| **Cape Cod** | Self-adjusts | ECR derived from data; external ELR change has no direct effect |

### If LDF / CDF Increases:

| Method | Impact | Mechanism |
|---|:---:|---|
| **Chain Ladder** | Increases | CL = Reported × CDF; directly proportional |
| **BF** | Increases (partial) | Higher CDF → higher % Unreported → higher BF IBNR; but effect is smaller than CL |
| **Benktander** | Increases (moderate) | Inherits from BF + more weight on development |
| **Cape Cod** | Increases (partial) | Higher CDF → lower % Reported → lower used-up premium → higher Cape Cod ECR |

### If Reported Losses Increase (e.g., large loss event):

| Method | Impact | Mechanism |
|---|:---:|---|
| **Chain Ladder** | Increases materially | CL = Reported × CDF; amplified by large CDF |
| **BF** | Increases slightly | BF adds reported directly; does NOT amplify through CDF |
| **Benktander** | Increases moderately | More weight on reported than pure BF |
| **Cape Cod** | Increases slightly (ECR rises) | Higher reported → higher Cape Cod ECR → higher IBNR |

### Exam-Ready Phrase:
> “BF does not project reported losses — it projects **unreported** losses using the a priori expected loss. A large reported loss enters BF directly (dollar-for-dollar) but is not amplified by the CDF, unlike Chain Ladder.”

### Maturity Effect Summary:

| Situation | Best Method | Reason |
|---|:---:|---|
| AY at ≤24 months (% paid ≤30%) | BF or Benktander | CL leverage is extreme; a priori more reliable |
| AY at 36–60 months | Benktander | Balances credibility; reduces ELR sensitivity |
| AY at ≥60 months (% paid ≥80%) | Chain Ladder | Development credible; a priori adds little value |
| ELR unreliable | Chain Ladder | BF/Benktander errors flow directly through a priori |
| Rapid growth / new line | BF | No development history; a priori is only anchor |
| Large loss distortion | BF or Benktander | Avoids CDF amplification of one-time event |


---
## BF and Benktander Assumptions — Exam Reference

Pure conceptual question type. Know each assumption and what happens when it’s violated.

### Bornhuetter-Ferguson Assumes:

| Assumption | What Happens if Violated |
|---|---|
| **ELR is reasonable and credible** | If ELR is wrong, error flows directly into IBNR; impact = % Unreported × ELR error; worst for immature AYs |
| **Development pattern (CDF) correctly selected** | If CDF is wrong, % unreported is wrong; BF IBNR is misstated |
| **Loss emergence consistent across AYs** | If mix or operations changed, prior pattern may not transfer |
| **Premium on consistent rate basis** | If rate levels changed, ELR applied to unadjusted premium misstates expected ultimate |

### Benktander Adds One More Assumption:
- **The BF ultimate is a reasonable intermediate prior**: Benktander replaces the a priori ELR with the BF ultimate after one iteration. If BF ultimate is itself biased (from poor ELR), Benktander inherits that bias (though attenuated).

### Cape Cod Assumes:

| Assumption | What Happens if Violated |
|---|---|
| **Premium is on consistent on-level basis** | If rate changes not adjusted for, Cape Cod ECR is distorted |
| **Sufficient credible reported claims** | If data sparse, self-derived ECR is unreliable |
| **Development pattern (CDF) correct** | Used-up premium and % reported depend on CDF accuracy |

### Key Comparison: BF vs. CL Assumptions

| | Chain Ladder | BF / Benktander |
|---|---|---|
| **Trusts** | Historical development pattern (100%) | A priori ELR (partially) |
| **Ignores** | A priori expected loss | Implied development signal from reported |
| **Assumption about data** | Development pattern is stable and representative | ELR is credible and current |
| **Sensitive to** | LDF selection errors (amplified by CDF) | ELR selection errors (weighted by % unreported) |

### Exam-Ready Answer Template:
> “BF is appropriate here because [specific situation: immature AY / rapid growth / volatile development] means the Chain Ladder development pattern is not credible. BF relies on the a priori ELR of [X%], which is more reliable than the sparse reported data. The key assumption is that the ELR is credible — if it is significantly wrong, BF IBNR will be [overstated / understated] by approximately % unreported × ELR error.”


---
## Exam-Style Written Answer Examples — BF and Benktander

### Scoring Reminder
Examiners penalize: missing maturity context, no direction of impact, generic reasoning. They reward: clear credibility logic, explicit mention of maturity / leverage, structured reasoning.

---

### Example A — Why Use BF Instead of Chain Ladder?

**Question:** “For AY 2025 (12 months of development), explain why you would use the Bornhuetter-Ferguson method instead of the Chain Ladder.” *(3 points)*

**Weak answer (1/3 points):**
> “BF is more stable for immature years.”

**Strong answer (3/3 points):**
> “At 12 months, AY 2025 has only a small fraction of losses reported. The Chain Ladder applies a CDF of 4.0× to this sparse data, meaning any LDF selection error is amplified four-fold into the ultimate estimate — this leverage makes CL unreliable.
>
> BF avoids this problem by projecting the unreported portion using the a priori ELR rather than development factors. Since 75% of losses are unreported, BF places 75% weight on the ELR assumption and only 25% on actual reported claims, producing a much more stable estimate.
>
> The caveat is that BF’s reliability depends on ELR quality: if the ELR is wrong by 10%, BF IBNR is wrong by approximately 7.5% (= 75% weight × 10% ELR error).”

**Why it works:** Names the leverage mechanism, quantifies the weight, and acknowledges the ELR risk with a direction.

---

### Example B — Why Benktander Over BF?

**Question:** “Explain the Benktander method and why it may be preferred over BF for an AY at 36 months of development.” *(3 points)*

**Strong answer:**
> “Benktander is an iterative method that uses the BF ultimate as its own prior in a second application of the BF formula: BK = Reported + % Unreported × BF Ultimate. Because BF Ultimate already incorporates some actual claims data (the reported losses appear directly in BF Ultimate), Benktander gives slightly more credibility to actual experience than pure BF.
>
> At 36 months, the AY has meaningful reported claims (perhaps 50–60% paid). Benktander is preferred over BF because it reduces dependence on the a priori ELR without swinging fully to Chain Ladder, which is still volatile at this maturity. It is a more balanced credibility blend.
>
> Common trap: Benktander is not a 50/50 average of CL and BF. The weight applied is always % unreported (= 1 − 1/CDF).”

---

### Example C — Impact of ELR Change

**Question:** “The selected ELR increases from 0.72 to 0.78. Describe the impact on the BF, Benktander, and Chain Ladder ultimates.” *(3 points)*

**Strong answer:**
> “Chain Ladder is unaffected — it does not use ELR at all.
>
> BF ultimate increases by % Unreported × (0.06 × Earned Premium) for each AY. Immature AYs with high % unreported are most affected; mature AYs with low % unreported are barely impacted.
>
> Benktander increases as well, but by a smaller amount. Because Benktander uses BF Ultimate (not raw ELR × Premium) as its prior, the ELR change is attenuated through one iteration — the effect is approximately (% Unreported)$^2$ × ΔELR × Premium.”

---

### Example D — Select and Justify a Method (Classic 7-Point Question)

**Question:** “You have computed CL, BF, and Benktander ultimates. For the most recent accident year (at 12 months), select one method and justify in 3–4 sentences.” *(3 points)*

**Strong answer:**
> “I recommend Benktander for the most recent accident year.
>
> Chain Ladder is inappropriate because the CDF at 12 months is very large (typically 3–5×), creating extreme leverage that amplifies any LDF selection error; sparse early claims data is not sufficient to drive a reliable CL estimate.
>
> Pure BF is reasonable but places 100% weight on the a priori ELR for the unreported portion, discarding any partial signal from actual reported claims.
>
> Benktander provides a better balance: it uses the BF ultimate as its prior, giving more credibility to actual experience than pure BF while still avoiding the full leverage of Chain Ladder. It is the most appropriate method for an immature year with a credible ELR available.”

---

### Key Phrases Examiners Reward:
- *“…CL applies a [X]× CDF to sparse data — leverage amplifies any LDF error…”*
- *“…BF places [% unreported] weight on the a priori ELR…”*
- *“…Benktander is NOT a simple average of CL and BF…”*
- *“…as development matures, Benktander converges to Chain Ladder…”*
- *“…if ELR is wrong, BF error ≈ % Unreported × ELR error…”*
- *“…Cape Cod is self-calibrating — it derives ECR from the data itself…”*


---
## Exam Trap Awareness — BF, Benktander, and Cape Cod

From CAS Examiner Reports and general exam patterns.

### Mechanical Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **% Unreported = 1 − CDF** | Using CDF instead of 1/CDF | % Unreported = 1 − 1/CDF; CDF is always ≥1.0; % Unreported is always <1 |
| **ELR × Reported** | Multiplying ELR by reported losses instead of earned premium | Expected Ultimate = ELR × **Earned Premium**, always |
| **Forgetting IBNR = Ult − Reported** | Reporting BF Ultimate as IBNR | IBNR = BF Ultimate − Reported; equivalently, IBNR = Expected Ultimate × % Unreported |
| **Benktander = average of CL and BF** | Using 50/50 blend instead of % unreported weight | BK = Reported + % Unreported × BF Ultimate; weight is always % unreported |
| **Forgetting tail factor in CDF** | CDF chain doesn’t include tail; % unreported is wrong | Always confirm whether a tail is given and multiply into CDF before computing % unreported |
| **Cape Cod ECR from unadjusted premium** | On-level adjustment ignored; ECR biased | Always ask: have rate levels changed? If yes, on-level premium before computing Cape Cod ECR |
| **ECR from immature AYs** | Cape Cod ECR biased downward if immature AYs included | Used-up premium weights immature AYs less naturally, but check calibration |

### Judgment / Written Answer Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **No maturity context** | Recommending a method without naming % paid or development age | Always state the AY’s maturity: “at 12 months, % paid = 25%…” |
| **No ELR risk caveat** | Recommending BF without noting ELR sensitivity | Always pair BF recommendation with: “if ELR is wrong by X%, BF error ≈ % unreported × X%” |
| **No direction of impact** | Saying ELR change affects reserves without specifying direction | Always say: higher ELR → higher BF ultimate (and quantify if possible) |
| **Confusing BF and EC method** | Treating them as the same thing | EC = 100% prior (ignores reported entirely); BF = reported + % unreported × expected |
| **Not explaining convergence** | Saying Benktander converges to CL without explaining why | As iterations increase, prior becomes more and more data-driven until it matches CL |
| **Generic answers** | “BF is stable” with no further detail | Stable against what? LDF volatility, large loss distortion, sparse data — name the specific risk |

---

### Quick Self-Check Before Finalizing Any BF/Benktander Answer:

- [ ] Did I compute **% Unreported = 1 − 1/CDF** (not 1 − CDF)?
- [ ] Did I compute **Expected Ultimate = ELR × Earned Premium** (not × Reported)?
- [ ] Did I compute **BF Ultimate = Reported + Expected Ultimate × % Unreported**?
- [ ] Did I compute **IBNR = BF Ultimate − Reported** (not just Expected Ultimate)?
- [ ] For Benktander: did I use **BF Ultimate** as the prior (not raw ELR × Premium)?
- [ ] Did I include the **tail factor** in the CDF before computing % unreported?
- [ ] If recommending BF — did I **name the maturity** and **acknowledge ELR risk**?
- [ ] If comparing methods — did I state which is highest/lowest and **explain why**?


---
## Method Comparison Quick Reference — What You Must Be Able to Do Instantly

### Core Formula Reference

| Method | Formula | Key Input |
|---|---|---|
| **Chain Ladder** | Ultimate = Reported × CDF | Historical development pattern |
| **Expected Claims** | Ultimate = ELR × Premium | A priori ELR only |
| **BF** | Ultimate = Reported + (1 − 1/CDF) × (ELR × Premium) | Both: ELR + CDF |
| **Benktander** | Ultimate = Reported + (1 − 1/CDF) × BF Ultimate | Both: ELR + CDF + iteration |
| **Cape Cod** | ECR = ΣReported / Σ(Premium × % Paid); Ultimate = Reported + (1 − 1/CDF) × (ECR × Premium) | Self-calibrating ECR |

### Method Sensitivity Matrix

| Situation | CL | BF | Benktander | Cape Cod |
|---|:---:|:---:|:---:|:---:|
| **Immature AY (≤24 mo)** | Volatile | Stable | Moderate | Stable |
| **Mature AY (≥60 mo)** | Strong | Conservative | Near-CL | Near-CL |
| **Rapid premium growth** | Risky | Safer | Safer | Safer |
| **ELR unreliable** | Unaffected | Weak | Moderate risk | Self-calibrates |
| **Large loss distortion** | Amplified | Dampened | Moderate | Dampened |
| **Stable dev. pattern** | Strong | No advantage | No advantage | No advantage |
| **No premium data** | OK | Needs premium | Needs premium | Cannot use |

### Convergence Rules (must know):
```
Iteration 0:   Benktander = Expected Claims (100% a priori)
Iteration 1:   Benktander ≈ BF
Iteration 2:   Benktander = Benktander (standard exam version)
Iteration n:   Benktander → Chain Ladder (as n → ∞)

As % Paid → 0%:   BF → Expected Claims
As % Paid → 100%: BF → Chain Ladder (CDF → 1.000)
```

### Classic Exam Question Structures:

**4–5 point mechanical:**
```
(a) Compute BF ultimate for all AYs          [2 pts]
(b) Compute Benktander ultimate for all AYs  [2 pts]
(c) Explain difference for most immature AY  [1 pt]
```

**6–7 point comparison + judgment:**
```
(a) Compute CL ultimate                  [2 pts] -- mechanical
(b) Compute BF ultimate                  [2 pts] -- mechanical
(c) Select method for most recent AY     [2 pts] -- judgment
(d) What changes if ELR increases?       [1 pt]  -- directional
```

Part (c) and (d) are where most points are lost. Strong answers always:
1. Name the AY’s **maturity** (% paid, development age)
2. Name the **leverage risk** of CL at that maturity
3. Name the **ELR sensitivity** of BF
4. Explain why Benktander is the **balanced middle ground**
5. For sensitivity: state **which methods are affected** and in **which direction**


---
## Practice Problem 4: Cape Cod — Mechanical Calculation

**Exam-style question (mechanical, 5–6 points):**

An actuary is using the Cape Cod method to estimate reserves. On-level earned premiums and reported losses are provided below.

| AY   | On-Level Premium (000s) | Reported to Date (000s) | CDF to Ultimate |
|------|------------------------:|------------------------:|----------------:|
| 2021 | 50,000                  | 37,500                  | 1.000           |
| 2022 | 44,000                  | 30,000                  | 1.100           |
| 2023 | 25,000                  | 15,000                  | 1.250           |
| 2024 | 32,000                  | 15,000                  | 1.600           |
| 2025 | 40,000                  | 7,500                   | 4.000           |

**(a)** For each AY, compute % Developed (= 1/CDF) and Used-Up Premium (= On-Level Premium × % Developed). *(2 pts)*  
**(b)** Compute the Cape Cod ELR = Total Reported / Total Used-Up Premium. *(1 pt)*  
**(c)** Compute Cape Cod IBNR for each AY (= Cape Cod ELR × On-Level Premium × % Unreported). *(2 pts)*  
**(d)** Compute Cape Cod Ultimate for each AY. *(1 pt)*  

> **Common traps:**  
> (1) % Developed = **1/CDF**, not CDF itself and not (1 − 1/CDF).  
> (2) Use **on-level** premium, not raw premium.  
> (3) ELR applies to **total on-level premium**, not just used-up premium, for the IBNR calc.


In [ ]:
import pandas as pd

# PRACTICE PROBLEM 4 SOLUTION: Cape Cod Calculation
data = {
    'AY':       [2021,  2022,  2023,  2024,  2025],
    'Premium':  [50000, 44000, 25000, 32000, 40000],
    'Reported': [37500, 30000, 15000, 15000,  7500],
    'CDF':      [1.000, 1.100, 1.250, 1.600, 4.000],
}
df = pd.DataFrame(data).set_index('AY')

# PART (a): % Developed = 1/CDF;  Used-Up Premium = Premium x % Developed
df['Pct_Developed']  = 1 / df['CDF']
df['Used_Up_Prem']   = df['Premium'] * df['Pct_Developed']
df['Pct_Unreported'] = 1 - df['Pct_Developed']

# PART (b): Cape Cod ELR = Total Reported / Total Used-Up Premium
cc_elr = df['Reported'].sum() / df['Used_Up_Prem'].sum()

# PART (c): Cape Cod IBNR = CC_ELR x On-Level Premium x % Unreported
df['CC_IBNR']    = cc_elr * df['Premium'] * df['Pct_Unreported']

# PART (d): Cape Cod Ultimate = Reported + IBNR
df['CC_Ultimate'] = df['Reported'] + df['CC_IBNR']

# Build results table
result = df[['Premium', 'Reported', 'CDF', 'Pct_Developed',
             'Used_Up_Prem', 'Pct_Unreported', 'CC_IBNR', 'CC_Ultimate']].copy()
result.loc['Total'] = {
    'Premium':       df['Premium'].sum(),
    'Reported':      df['Reported'].sum(),
    'CDF':           float('nan'),
    'Pct_Developed': float('nan'),
    'Used_Up_Prem':  df['Used_Up_Prem'].sum(),
    'Pct_Unreported':float('nan'),
    'CC_IBNR':       df['CC_IBNR'].sum(),
    'CC_Ultimate':   df['CC_Ultimate'].sum(),
}

print('=== PRACTICE PROBLEM 4: CAPE COD SOLUTION ===')
print(f'  Cape Cod ELR = Total Reported / Total Used-Up Premium')
print(f'               = {df["Reported"].sum():,.0f} / {df["Used_Up_Prem"].sum():,.0f}'
      f' = {cc_elr:.4f}')
print()
print(result.to_string(
    float_format=lambda x: f'{x:.4f}' if 0 < abs(x) < 10 else f'{x:,.0f}'
))
print()

# Show the weighting insight
print('=== WEIGHTING INSIGHT ===')
print('More mature AYs (higher % Developed) get more weight in the ELR:')
print(f'  {"AY":>5}  {"% Dev":>8}  {"Used-Up":>10}  {"Weight":>8}')
total_uup = df['Used_Up_Prem'].sum()
for ay in df.index:
    uup = df.loc[ay, 'Used_Up_Prem']
    pct = df.loc[ay, 'Pct_Developed']
    print(f'  {ay:>5}  {pct:>8.4f}  {uup:>10,.0f}  {uup/total_uup*100:>7.1f}%')
print(f'  {"Total":>5}  {"":>8}  {total_uup:>10,.0f}  {100.0:>7.1f}%')
print()
print(
    'KEY CHECKS:\n'
    '  (1) AY 2021 (CDF=1.000, fully developed) has the HIGHEST weight\n'
    '      because % Developed = 100% -- this is correct and expected.\n'
    '  (2) AY 2025 has the LOWEST weight (% Developed = 25%) --\n'
    '      sparse data from immature years does not dominate the ELR.\n'
    '  (3) CC_ELR x Total Premium = ELR x Used-Up + ELR x Unreported Premium\n'
    f'      = {cc_elr:.4f} x {df["Premium"].sum():,} = {cc_elr*df["Premium"].sum():,.0f}\n'
    f'      = Total Ultimate of {df["CC_Ultimate"].sum():,.0f} (matches)'
)


---
## Cape Cod — Weighting Structure and Conceptual Reference

### Why Cape Cod Is Self-Calibrating

The Cape Cod ELR is a **credibility-weighted average of implied historical ELRs**, where the weight for each AY is its used-up premium:

$$
\text{CC ELR} = \frac{\sum_w \text{Reported}_w}{\sum_w (\text{On-Level Premium}_w \times \frac{1}{\text{CDF}_w})}
$$

**What this means in practice:**
- A **mature AY** (% developed = 90%) contributes 90% of its premium to the denominator — high weight, high credibility.
- An **immature AY** (% developed = 25%) contributes only 25% of its premium — low weight, low credibility.
- Immature year volatility is automatically muted in the ELR estimate.

### Cape Cod vs. Chain Ladder vs. BF

| Dimension | Chain Ladder | BF | Cape Cod |
|---|:---:|:---:|:---:|
| **ELR source** | Not used | External (pricing / management) | Derived internally from data |
| **Premium used?** | No | Yes | Yes |
| **ELR updates with new data?** | N/A | No (fixed a priori) | Yes (self-calibrates) |
| **Immature AY stability** | Low | High | High |
| **Sensitive to ELR error** | No | Yes | Partially (ELR from data, but can be biased) |
| **Sensitive to premium distortion** | No | No | **Yes — largest source of error** |
| **Best for** | Mature, stable AYs | New line / no internal ELR | Growing book with reliable premium |

### When Cape Cod Is Stronger Than BF:
- No reliable **external ELR** is available (e.g., new product, no pricing assumption)
- The book is **growing rapidly** — Chain Ladder overstates because immature AYs carry full CDF leverage; Cape Cod spreads losses over credibility-weighted premium
- Development is **volatile** — Cape Cod avoids relying on unstable early LDFs

### When Cape Cod Is Weaker:
- **Premium is not on consistent rate/coverage level** — derived ELR is biased (this is the #1 source of Cape Cod error on the exam)
- **Rate changes not adjusted** — used-up premium is distorted; ELR is wrong
- **Exposure base unstable** — mix changes mean premium does not represent comparable risk
- **Insufficient credible reported data** — self-derived ELR becomes unreliable

### Relationship Between Cape Cod and BF:

Both use the same IBNR formula: `IBNR = ELR × Premium × % Unreported`

The **only difference** is where the ELR comes from:
- **BF:** ELR is selected externally (pricing, management assumption)
- **Cape Cod:** ELR is derived from reported losses and used-up premium

> Exam-ready phrase: “Cape Cod is a self-calibrating version of BF. It uses the same projection formula but derives the ELR from current loss experience rather than an external assumption.”


---
## Rate Change Quick Check — Premium Treatment Deep Dive

**This is where most Cape Cod exam points are lost.** If premium is not adjusted to a consistent rate level, the Cape Cod ELR is biased.

### The Core Logic

Cape Cod ELR = Total Reported / Total Used-Up Premium

| If rates were… | Raw premium in earlier years is… | Used-up premium denominator is… | ELR is… | Projected ultimates are… |
|---|:---:|:---:|:---:|:---:|
| **Increasing** (and not on-leveled) | Too low | Too low | Overstated | Overstated |
| **Decreasing** (and not on-leveled) | Too high | Too high | Understated | Understated |

### Worked Example — Decreasing Rates

Suppose rates dropped 20% from AY 2023 to AY 2025. Raw (unadjusted) premiums and on-level premiums are:

| AY   | Raw Premium | On-Level Factor | On-Level Premium | % Dev | Used-Up (On-Level) | Used-Up (Raw) |
|------|------------:|:---------------:|-----------------:|------:|-------------------:|--------------:|
| 2023 | 100,000     | 1.250           | 125,000          | 80%   | 100,000            | 80,000        |
| 2024 | 90,000      | 1.111           | 100,000          | 60%   | 60,000             | 54,000        |
| 2025 | 80,000      | 1.000           | 80,000           | 25%   | 20,000             | 20,000        |
| **Total** |         |                 |                  |       | **180,000**        | **154,000**   |

Suppose Total Reported = 117,000.

**With on-level premium (correct):**  
CC ELR = 117,000 / 180,000 = **0.650**

**With raw premium (incorrect — rates decreasing):**  
CC ELR = 117,000 / 154,000 = **0.760** ← overstated!

Wait — decreasing rates means earlier years’ raw premium is **too high**. But on-leveling raises them further to current level? No — let’s re-examine:

If rates **decreased** over time (AY 2023 had higher rates than current):  
- AY 2023 raw premium was **higher** than a current-rate equivalent  
- To bring to current rate level, **divide by** cumulative rate increase (or apply factor < 1)  
- On-level premium for AY 2023 = Raw × (Current Index / AY 2023 Index) < Raw

Let’s revise with a concrete decreasing-rate scenario:

| AY   | Raw Premium | Cum. Rate Change | On-Level Factor | On-Level Premium |
|------|------------:|-----------------:|:---------------:|-----------------:|
| 2023 | 120,000     | +20% (higher)    | 0.833           | 100,000          |
| 2024 | 110,000     | +10%             | 0.909           | 100,000          |
| 2025 | 100,000     | 0% (base)        | 1.000           | 100,000          |

% Developed: 80%, 60%, 25% (same as before)  
**On-Level Used-Up:** 80,000 + 60,000 + 25,000 = 165,000  
**Raw Used-Up:** 96,000 + 66,000 + 25,000 = 187,000  

Total Reported = 110,000 (given)

**With on-level (correct):** ELR = 110,000 / 165,000 = **0.667**  
**With raw premium (incorrect):** ELR = 110,000 / 187,000 = **0.588** ← understated!

**Consequence:** Lower ELR → Lower projected ultimates → IBNR is understated.

### Exam-Ready Answer Template:
> “If rates were **decreasing** and premium is not on-leveled, earlier accident years carry **too-high raw premium** in the used-up premium denominator. This inflates the denominator relative to the true exposure, causing the Cape Cod ELR to be **understated**. Applying this understated ELR projects **too-low IBNR** for immature accident years. The fix is to on-level all premiums to the current rate level before computing the Cape Cod ELR.”

### Quick Check Summary:

| Scenario | Effect on Used-Up Premium | Effect on ELR | Effect on IBNR |
|---|:---:|:---:|:---:|
| Rates increasing, not on-leveled | Understated | Overstated | Overstated |
| Rates decreasing, not on-leveled | Overstated | Understated | Understated |
| Premium growing (volume), rates flat | No bias | No bias | Correct |
| Coverage changes not adjusted | Distorted | Distorted | Distorted |

**Key takeaway:** Cape Cod errors almost always come from mishandling premium, not development math. If the exam gives rate change information and asks for Cape Cod — always on-level the premium first.


---
## Cape Cod Exam Trap Awareness — Focused Reference

The following traps are specific to Cape Cod and supplement the general BF/Benktander trap table above.

### Mechanical Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **Used-up premium = Premium × CDF** | Multiplying by CDF (> 1.0) inflates the denominator | Used-Up = Premium × **(1/CDF)** = Premium × % Developed |
| **ELR denominator = Total Premium** | Using full premium instead of used-up | ELR = Reported / **(Used-Up Premium)**; used-up < total premium |
| **Ignoring rate changes** | Using raw nominal premium in a period with changing rates | Always ask: did rates change? If yes, on-level before computing ELR |
| **Mixing on-level and raw premium** | Some years adjusted, some not | Apply on-level factor consistently to all AYs |
| **IBNR applied to used-up premium** | Computing IBNR = ELR × Used-Up × % Unreported | IBNR = ELR × **Full On-Level Premium** × % Unreported (not used-up) |
| **Forgetting tail in CDF** | % Developed is too high; used-up premium is inflated | Always confirm whether a tail is included in CDF before computing 1/CDF |

### Judgment / Written Answer Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **CC = BF** | Treating Cape Cod and BF as identical methods | They use the same IBNR formula but differ in ELR source: BF uses external ELR; CC derives it internally |
| **ELR direction wrong** | Saying decreasing rates overstate ELR (it’s the opposite) | Decreasing rates → earlier years had higher rates → raw premium too high → denominator inflated → ELR understated |
| **No premium adjustment mentioned** | Recommending Cape Cod without noting the on-level requirement | Always caveat: “Cape Cod requires premium on a consistent rate/coverage basis” |
| **Not explaining weighting** | Saying CC uses “weighted premium” without explaining why | Immature AYs have low % developed → low used-up premium → low weight in ELR. Mature years dominate |

### Cape Cod vs. BF Written Answer Comparison:

**When asked “Why use Cape Cod instead of BF?” — strong answer:**
> “Cape Cod is preferred when no reliable external ELR is available. It derives the ELR directly from reported losses and used-up premium, giving more weight to mature accident years that are nearly fully developed. Unlike BF, it does not require an independent pricing assumption, making it more objective when experience data is available.”

**When asked “Why use BF instead of Cape Cod?” — strong answer:**
> “BF is preferred when premium data is unreliable or when rate/coverage changes make on-leveling difficult. A credible external ELR from pricing or benchmarks avoids the distortions that can arise in Cape Cod’s internally-derived ELR. BF also works on lines with no historical premium data.”
